In [25]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Concatenate,
    Flatten
)

from tensorflow.keras.models import Model

In [26]:
route_dataset_df = pd.read_pickle(
    "../data/processed/route_dataset.pkl"
)

print(route_dataset_df.shape)
route_dataset_df.head()

(19647, 15)


,Route ID,Driver ID,Country,Week ID,Day of Week,Planned Route,Actual Route,Planned Stop Count,Actual Stop Count,Planned Distance,Actual Distance,RQS,EDS,DDS,Label
0,0,0,1,0,Monday,"[0, 1, 2, 3, 3, 0, 1]","[0, 3, 3, 0, 1, 2, 1]",7,7,49.468094,44.965197,0.5000,0.5714,-0.0910,D
1,1,1,1,0,Monday,"[0, 4, 5, 6, 7, 8, 9]","[0, 9, 4, 7, 6, 5, 8]",7,7,33.274342,33.610418,0.5000,0.5714,0.0101,D
2,2,2,1,0,Monday,"[0, 10, 11, 12, 13, 14, 15]","[0, 12, 13, 11, 10, 14, 15]",7,7,12.124804,12.508786,0.6667,0.5714,0.0317,D
3,3,3,1,0,Monday,"[0, 16, 17, 18, 19, 20, 21, 22, 20, 23]","[0, 16, 19, 17, 18, 20, 22, 20, 21, 23]",10,10,19.039848,19.374644,0.8400,0.4000,0.0176,D
4,4,4,1,0,Monday,"[0, 24, 25, 26, 27, 28, 29, 30]","[0, 24, 30, 26, 27, 28, 29, 25]",8,8,20.632674,19.528799,0.6875,0.2500,-0.0535,D


In [27]:
route_dataset_df["Label"] = (
    route_dataset_df["Label"]
    .map({
        "ND": 0,
        "D": 1
    })
    .astype(np.int32)
)

print(route_dataset_df["Label"].value_counts())

Label
1    12158
0     7489
Name: count, dtype: int64


In [28]:
MAX_ROUTE_LENGTH = route_dataset_df[
    "Planned Route"
].apply(len).max()

print(MAX_ROUTE_LENGTH)

36


In [29]:
route_sequences = pad_sequences(
    route_dataset_df["Planned Route"],
    maxlen=MAX_ROUTE_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

print(route_sequences.shape)

(19647, 36)


In [30]:
max_stop_id = max([
    max(route)
    for route in route_dataset_df["Planned Route"]
])

VOCAB_SIZE = max_stop_id + 1

print(VOCAB_SIZE)

10864


In [31]:
driver_input = route_dataset_df[
    "Driver ID"
].values

numerical_features = route_dataset_df[
    [
        "Country",
        "Week ID",
        "Planned Stop Count",
        "Planned Distance"
    ]
].values

y = route_dataset_df["Label"].values

print(driver_input.shape)
print(numerical_features.shape)
print(y.shape)

(19647,)
(19647, 4)
(19647,)


In [32]:
driver_train, driver_test, \
route_train, route_test, \
num_train, num_test, \
y_train, y_test = train_test_split(
    driver_input,
    route_sequences,
    numerical_features,
    y,
    test_size=0.2,
    random_state=42
)

print(driver_train.shape)
print(route_train.shape)
print(num_train.shape)
print(y_train.shape)

(15717,)
(15717, 36)
(15717, 4)
(15717,)


In [33]:
driver_layer = Input(
    shape=(1,),
    name="Driver_Input"
)

driver_embedding = Embedding(
    input_dim=5000,
    output_dim=32
)(driver_layer)

driver_embedding = Flatten()(
    driver_embedding
)

route_layer = Input(
    shape=(MAX_ROUTE_LENGTH,),
    name="Route_Input"
)

route_embedding = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=64
)(route_layer)

route_lstm = LSTM(
    64
)(
    route_embedding
)

numerical_layer = Input(
    shape=(4,),
    name="Numerical_Input"
)

merged = Concatenate()(
    [
        driver_embedding,
        route_lstm,
        numerical_layer
    ]
)

dense = Dense(
    64,
    activation="relu"
)(
    merged
)

dense = Dense(
    32,
    activation="relu"
)(
    dense
)

output = Dense(
    1,
    activation="sigmoid"
)(
    dense
)

model = Model(
    inputs=[
        driver_layer,
        route_layer,
        numerical_layer
    ],
    outputs=output
)

In [20]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()
print(model)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Driver_Input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Route_Input         │ (None, 36)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 32)     │     12,800 │ Driver_Input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 36, 64)    │    695,296 │ Route_Input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 36)        │          0 │ Route_Input[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Numerical_Input     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     33,024 │ embedding_3[0][0… │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │        160 │ Numerical_Input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128)       │          0 │ flatten_1[0][0],  │
│ (Concatenate)       │                   │            │ lstm_1[0][0],     │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │     16,512 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         65 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 766,113 (2.92 MB)

 Trainable params: 766,113 (2.92 MB)

 Non-trainable params: 0 (0.00 B)

<Functional name=functional_1, built=True>


In [34]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Driver_Input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Route_Input         │ (None, 36)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 1, 32)     │    160,000 │ Driver_Input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 36, 64)    │    695,296 │ Route_Input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 32)        │          0 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     33,024 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Numerical_Input     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 100)       │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ lstm_2[0][0],     │
│                     │                   │            │ Numerical_Input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      6,464 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 32)        │      2,080 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         33 │ dense_9[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 896,897 (3.42 MB)

 Trainable params: 896,897 (3.42 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
history = model.fit(
    [
        driver_train,
        route_train,
        num_train
    ],
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=512,
    verbose=1
)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.5354 - loss: 1.3192 - val_accuracy: 0.5856 - val_loss: 0.7186
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.6472 - loss: 0.6380 - val_accuracy: 0.6676 - val_loss: 0.6036
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.6979 - loss: 0.5764 - val_accuracy: 0.7204 - val_loss: 0.5552
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.7391 - loss: 0.5325 - val_accuracy: 0.7510 - val_loss: 0.5101
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.7651 - loss: 0.4947 - val_accuracy: 0.7567 - val_loss: 0.4885
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.7724 - loss: 0.4749 - val_accuracy: 0.7595 - val_loss: 0.4824
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.7724 - loss: 0.4711 - val_accuracy: 0.7545 - val_loss: 0.4810
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.7713 - loss: 0.4675 - val_accuracy: 0.7576 - v

In [36]:
y_prob = model.predict(
    [
        driver_test,
        route_test,
        num_test
    ]
)

y_pred = (
    y_prob > 0.5
).astype(int)

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [37]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("="*60)
print("LSTM CLASSIFICATION RESULTS")
print("="*60)

print(f"Accuracy : {accuracy:.4f}")

print()

print(
    classification_report(
        y_test,
        y_pred
    )
)

print("Confusion Matrix:\n")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

LSTM CLASSIFICATION RESULTS
Accuracy : 0.7644

              precision    recall  f1-score   support

           0       0.70      0.66      0.68      1500
           1       0.80      0.83      0.81      2430

    accuracy                           0.76      3930
   macro avg       0.75      0.74      0.75      3930
weighted avg       0.76      0.76      0.76      3930

Confusion Matrix:

[[ 993  507]
 [ 419 2011]]


In [38]:
model.save(
    "../outputs/models/lstm_classification.keras"
)

print(
    "LSTM Classification model saved successfully!"
)

LSTM Classification model saved successfully!


In [24]:
print(route_dataset_df["Label"].unique())
print(route_dataset_df["Label"].dtype)

<StringArray>
['D', 'ND']
Length: 2, dtype: str
str
